In [ ]:
# print("Hellow!")

Hellow!


In [1]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [14]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

In [3]:
from ingest import load_faq_data, build_index

In [4]:
from rag_helper import RAGBase

In [9]:
model='gemini-3.5-flash'

In [5]:
documents = load_faq_data()
index = build_index(documents)

In [6]:
instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index = index,
    llm_client = client,
    instructions = instructions
)

In [28]:
answer = assistant.rag("How do I run Ollama locally?")
print(answer)

AttributeError: 'RAGBase' object has no attribute 'chat_session'

In [27]:
messages = [
    {'role': 'user', 'content': 'I just discovered the course. Can I join it?'}
]

response = client.models.generate_content(
        model=model,
        contents=messages
)

response.output_text

ValidationError: 16 validation errors for _GenerateContentParameters
contents.Content
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=[{'role': 'user', 'conten...ourse. Can I join it?'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/model_attributes_type
contents.str
  Input should be a valid string [type=string_type, input_value=[{'role': 'user', 'conten...ourse. Can I join it?'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type
contents.File
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=[{'role': 'user', 'conten...ourse. Can I join it?'}], input_type=list]
    For further information visit https://errors.pydantic.dev/2.13/v/model_attributes_type
contents.Part
  Input should be a valid dictionary or object to extract fields from [type=model_attributes_type, input_value=

In [29]:
def search(query):
    boost_dict = {'question': 3.0, 'section': 0.5}
    filter_dict = {'course': 'llm-zoomcamp'}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

In [37]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the FAQ database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course FAQ.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

In [35]:
messages = [
    types.Content(
        role="user", 
        parts=[types.Part.from_text(text="Hi, I am planning a trip to Tokyo.")]
    ),
    types.Content(
        role="model", 
        parts=[types.Part.from_text(text="That sounds exciting! Tokyo is amazing. How can I help you plan?")]
    ),
    types.Content(
        role="user", 
        parts=[types.Part.from_text(text="What are 3 must-visit places there? Use Google Search if you need up-to-date spots.")]
    )
]

In [39]:
response = client.models.generate_content(
    model='gemini-3.5-flash',
    contents=messages,
    tools=[search_tool]
)

TypeError: Models.generate_content() got an unexpected keyword argument 'tools'